# Reuse catalogue definitions in a custom archetype

Use the installed development revision of `epsm_data` with Python 3.11+. The example reads packaged registries/profiles and writes no files; it needs no Core, database, network or engine. Catalogue selection is a documented modelling assumption, not a measured observation. Rates retain their explicit area/volume bases.

In [ ]:
from epsm_data import Catalogue, Layer, LayerMaterial

catalogue = Catalogue.load()
print(catalogue)
archetype = catalogue.for_year("multi_family", 1970)
assert archetype.operating_profile.profile_id.startswith("sveby.residential")
assert archetype.hvac_template is None
assert archetype.envelope_performance.infiltration_l_s_m2 == 0.2
assert archetype.envelope_performance.infiltration_reference_basis == "Atemp"

Read-only indexes expose the existing physical objects. Materials can be reused directly in new layers; whole-window definitions remain separate from optical layer materials and opening geometry. The 0.15 m thickness below is an illustrative project assumption.

In [ ]:
concrete = catalogue.materials["concrete"]
layer = Layer("Project concrete", "project.layer", 0.15, (LayerMaterial(concrete, 1, 0.15),))
assert layer.materials[0].material.uuid == concrete.uuid
assert catalogue.windows["window_existing"].u_value > 0
assert catalogue.construction_sets()[("mf_se02", "existing")] == archetype.construction_set

A project copy uses the same class as a selected entry. No installed system is inferred from a construction era or ventilation classification. The generic operating option is available only through an explicit caller choice.

In [ ]:
project = archetype.with_changes(
    display_name="Project study",
    notes=archetype.notes + ("Explicit project selection for demonstration.",),
)
assert project.construction_set == archetype.construction_set
assert project.display_name != archetype.display_name
generic = catalogue.get("generic_se02")
assert generic.operating_profile is None
explicit = generic.with_changes(operating_profile=catalogue.operating_profiles["generic_other"])
assert explicit.operating_profile.notes
assert generic.operating_profile is None

SE.07 has one configuration beginning in 2022. Effective slot inspection retains existing classifications while physical compositions remain separately accessible. Complete custom/catalogue archetype exchange uses format 4.

In [ ]:
import json
from epsm_data import archetype_to_dict, archetype_from_dict

assert catalogue.era_for_year(2022) == "SE.07"
assert catalogue.renovation_states("mf_se07") == ()
slot = catalogue.get_slot("multi_family:SE.02:SE.Zone3:existing:F")
assert slot.infiltration.evidence
assert catalogue.effective_configuration()["dataset_id"] == catalogue.dataset_id
restored = archetype_from_dict(json.loads(json.dumps(archetype_to_dict(project))))
assert restored == project
print(project)

The pure construction handover keeps explicit source/boundary context. It does not assess calibration targets or resolve site-specific ground inputs.

In [ ]:
from epsm_data import export_nominal_constructions

bundle = export_nominal_constructions(
    archetype, metadata=catalogue.nominal_metadata("mf_se02", state="existing")
)
assert bundle["core"]["EXT_WALL"]
assert not bundle["readiness"]["ground_validation"]